
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Appendix - Agent Deployment Patterns

## Overview

This course uses `agents.deploy()` to deploy agents to Model Serving endpoints. While this approach works well for evaluation workflows, Databricks now recommends **Databricks Apps** as the deployment target for new agent projects.

This appendix provides a brief overview of the deployment landscape so you can make informed decisions when deploying your own agents.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">This appendix is reference material only. The evaluation techniques covered in this course apply regardless of which deployment method you choose.</p>
</div>

## A. Model Serving vs. Databricks Apps

<div style="display:flex;gap:16px;margin:18px 0;flex-wrap:wrap;font-family:sans-serif;">

<div style="flex:1;min-width:280px;border:2px solid #2272B4;border-radius:10px;padding:18px;background:#fff;">
<div style="font-weight:700;font-size:16px;color:#2272B4;margin-bottom:10px;">Model Serving (agents.deploy)</div>
<ul style="margin:0;padding-left:18px;font-size:14px;color:#333;line-height:1.8;">
<li>Single function call to deploy</li>
<li>Automatic infrastructure provisioning</li>
<li>Built-in inference tables and tracing</li>
<li>Review App for stakeholder feedback</li>
<li>Agent code logged as MLflow artifact</li>
<li>Limited control over server configuration</li>
</ul>
</div>

<div style="flex:1;min-width:280px;border:2px solid #00A972;border-radius:10px;padding:18px;background:#fff;">
<div style="font-weight:700;font-size:16px;color:#00A972;margin-bottom:10px;">Databricks Apps (Recommended)</div>
<ul style="margin:0;padding-left:18px;font-size:14px;color:#333;line-height:1.8;">
<li>Git-based development with DABs</li>
<li>Full control over code and server config</li>
<li>Built-in Chat UI and <code>/invocations</code> endpoint</li>
<li>Local development with <code>uv run start-app</code></li>
<li>Async FastAPI server (MLflow AgentServer)</li>
<li>Explicit auth via <code>databricks.yml</code></li>
</ul>
</div>

</div>

### When to use each

- **Model Serving** works well for quick prototyping, evaluation workflows (like this course), and existing deployments that don't need migration yet.
- **Databricks Apps** is the recommended path for new production deployments where you need Git-based CI/CD, local development, and full control over the serving infrastructure.

Both approaches support MLflow tracing and evaluation. The `mlflow.genai.evaluate()` patterns you learned in this course work identically against agents deployed via either method.

For full details, see: Author an AI agent and deploy it on Databricks Apps (<a href="https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent" target="_blank">AWS</a> | <a href="https://docs.databricks.com/azure/en/generative-ai/agent-framework/author-agent" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/agent-framework/author-agent" target="_blank">GCP</a>)

## B. Migrating from Model Serving to Databricks Apps

If you have an agent deployed via `agents.deploy()` and want to migrate to Databricks Apps, the process involves restructuring your code and configuration.

### Migration steps

<div style="display:flex;align-items:center;justify-content:center;gap:0;margin:18px 0;flex-wrap:nowrap;font-family:sans-serif;">
  <div style="background:#2272B4;color:#fff;padding:10px 14px;border-radius:8px 0 0 8px;font-weight:600;font-size:13px;text-align:center;min-width:120px;">1. Download<br/><span style="font-weight:400;font-size:12px;">Extract MLflow<br/>artifacts</span></div>
  <div style="color:#999;font-size:18px;padding:0 4px;">&#x2192;</div>
  <div style="background:#FF5F46;color:#fff;padding:10px 14px;font-weight:600;font-size:13px;text-align:center;min-width:120px;">2. Restructure<br/><span style="font-weight:400;font-size:12px;">Class methods to<br/>decorated functions</span></div>
  <div style="color:#999;font-size:18px;padding:0 4px;">&#x2192;</div>
  <div style="background:#00A972;color:#fff;padding:10px 14px;font-weight:600;font-size:13px;text-align:center;min-width:120px;">3. Configure<br/><span style="font-weight:400;font-size:12px;">Create<br/>databricks.yml</span></div>
  <div style="color:#999;font-size:18px;padding:0 4px;">&#x2192;</div>
  <div style="background:#1B5162;color:#fff;padding:10px 14px;border-radius:0 8px 8px 0;font-weight:600;font-size:13px;text-align:center;min-width:120px;">4. Deploy<br/><span style="font-weight:400;font-size:12px;">bundle deploy</span></div>
</div>

### What changes

| Aspect | Model Serving | Databricks Apps |
|---|---|---|
| **Agent code** | Class-based `ResponsesAgent` with `predict()` / `predict_stream()` | Module-level `@invoke()` and `@stream()` decorated functions |
| **Configuration** | MLmodel file + model config YAML | `databricks.yml` with resource declarations |
| **Execution model** | Synchronous | Async recommended (`async def`) |
| **Deployment command** | `agents.deploy()` | `databricks bundle deploy` |

### What stays the same

- Core agent logic (tool definitions, LLM calls, prompt templates)
- MLflow tracing and evaluation integration
- Unity Catalog tool registration
- Request/response payload format

For the full migration guide, see: Migrate an agent from Model Serving to Databricks Apps (<a href="https://docs.databricks.com/aws/en/generative-ai/agent-framework/migrate-agent-to-apps" target="_blank">AWS</a> | <a href="https://docs.databricks.com/azure/en/generative-ai/agent-framework/migrate-agent-to-apps" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/agent-framework/migrate-agent-to-apps" target="_blank">GCP</a>)

## C. The Agent Deployment Spectrum

Databricks provides multiple levels of abstraction for building and deploying agents:

<div style="display:flex;align-items:stretch;justify-content:center;gap:10px;margin:18px 0;flex-wrap:wrap;font-family:sans-serif;">
  <div style="flex:1;min-width:180px;background:#2272B4;color:#fff;padding:14px;border-radius:8px;text-align:center;">
    <div style="font-weight:700;font-size:15px;">Agent Bricks</div>
    <div style="font-size:12px;margin-top:6px;opacity:0.9;">Fully declarative.<br/>Knowledge Assistants,<br/>Supervisor Agents.<br/>No code required.</div>
  </div>
  <div style="flex:1;min-width:180px;background:#00A972;color:#fff;padding:14px;border-radius:8px;text-align:center;">
    <div style="font-weight:700;font-size:15px;">Supervisor API</div>
    <div style="font-size:12px;margin-top:6px;opacity:0.9;">Programmatic control.<br/>Databricks manages the<br/>agent loop. You define<br/>model, tools, instructions.</div>
  </div>
  <div style="flex:1;min-width:180px;background:#FF5F46;color:#fff;padding:14px;border-radius:8px;text-align:center;">
    <div style="font-weight:700;font-size:15px;">Databricks Apps</div>
    <div style="font-size:12px;margin-top:6px;opacity:0.9;">Full custom agents.<br/>Git-based, async,<br/>any framework.<br/>Recommended for production.</div>
  </div>
  <div style="flex:1;min-width:180px;background:#1B5162;color:#fff;padding:14px;border-radius:8px;text-align:center;">
    <div style="font-weight:700;font-size:15px;">AI Gateway APIs</div>
    <div style="font-size:12px;margin-top:6px;opacity:0.9;">Maximum control.<br/>Write your own agent loop.<br/>Databricks provides<br/>LLM inference only.</div>
  </div>
</div>

<div style="text-align:center;font-size:13px;color:#666;margin-top:4px;">← More managed &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; More control →</div>

Regardless of where your agent sits on this spectrum, the evaluation techniques from this course apply. MLflow tracing, scorers, and `mlflow.genai.evaluate()` work across all deployment patterns.

For more on the Supervisor API, see: Supervisor API (<a href="https://docs.databricks.com/aws/en/generative-ai/agent-bricks/supervisor-api" target="_blank">AWS</a> | <a href="https://docs.databricks.com/azure/en/generative-ai/agent-bricks/supervisor-api" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/agent-bricks/supervisor-api" target="_blank">GCP</a>)

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
